In [43]:
import os, math, scipy, skbio, statistics
import numpy as np
import pandas as pd
from skbio.diversity import beta_diversity
from skbio.stats.ordination import pcoa
from skbio.stats.composition import clr
from skbio.stats.composition import multiplicative_replacement
import seaborn as sns
import matplotlib as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import fdrcorrection

### Get profiles

In [2]:
abundances = pd.read_table('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_profiles.tsv')
abundances.index = abundances['Unnamed: 0']
abundances = abundances.drop('Unnamed: 0',1)
abundances

/tmp/ipykernel_126555/1478764157.py:3: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  abundances = abundances.drop('Unnamed: 0',1)


,MMRS62666492ST-27-0-0,MMRS51737257ST-27-0-0,MMRS38861560ST-27-0-0,MMRS95674036ST-27-0-0,MMRS67096717ST-27-0-0,MMRS84085284ST-27-0-0,MMRS51307502ST-27-0-0,MMRS17963147ST-27-0-0,MMRS39582183ST-27-0-0,MMRS35808664ST-27-0-0,...,SID15785561_SF07,SID19401504_SF05,SID20269046_SF08,SID50492523_SF07,SID10502288_SF08,SID72242320_SF07,SID30920067_SF07,SID11486372_SF08,SID16913125_SF05,SID16853657_SF05
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
k__Bacteria,99.93463,99.18388,100.00000,100.00000,100.00000,100.00000,97.99128,99.92570,93.18519,99.99433,...,100.00000,100.00000,99.20735,100.00000,99.62772,99.57517,99.92819,99.93030,99.53060,100.00000
k__Archaea,0.06537,0.81612,0.00000,0.00000,0.00000,0.00000,2.00872,0.07430,6.81481,0.00567,...,0.00000,0.00000,0.79265,0.00000,0.37228,0.42483,0.07181,0.06970,0.46940,0.00000
k__Bacteria|p__Firmicutes,69.66820,58.11043,69.86201,28.13507,54.76472,48.93404,81.31657,53.89673,70.39620,38.26343,...,62.99205,57.98725,77.37153,56.84940,62.01985,70.64182,57.19654,64.96287,72.69495,56.00511
k__Bacteria|p__Bacteroidetes,28.66924,37.02095,28.10208,66.97080,20.40542,48.32282,10.73218,42.41865,19.37239,51.40790,...,35.22573,38.97574,16.54528,35.27852,27.46251,21.14656,39.11211,20.96019,17.53030,37.48840
k__Bacteria|p__Proteobacteria,0.67858,1.44754,0.66079,2.82776,0.67835,0.02112,3.02764,2.76147,1.31107,9.69329,...,0.49921,0.86482,0.72299,3.39898,3.28930,2.96303,1.12058,11.84127,2.04371,0.62305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27883|s__GGB27883_SGB40317,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27883|s__GGB27883_SGB40317|t__SGB40317,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
k__Bacteria|p__Bacteroidetes|c__CFGB570|o__OFGB570|f__FGB570|g__GGB1201,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000


In [3]:
keep = list()
for x in abundances.index.tolist():
    keep.append('t__' in x)
sgb_abundances = abundances.loc[keep]
sgb_abundances = sgb_abundances.transpose()
sgb_abundances = sgb_abundances.drop(sgb_abundances.loc[sgb_abundances.sum(axis=1) == 0].index.tolist())
sgb_abundances

Unnamed: 0,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__GGB3226|s__GGB3226_SGB4260|t__SGB4260,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__Phocaeicola|s__Phocaeicola_vulgatus|t__SGB1814,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Rikenellaceae|g__Alistipes|s__Alistipes_putredinis|t__SGB2318,k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales|f__Veillonellaceae|g__Dialister|s__Dialister_invisus|t__SGB5825_group,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii|t__SGB15332_group,k__Bacteria|p__Firmicutes|c__Bacilli|o__Bacilli_unclassified|f__Bacilli_unclassified|g__Bacilli_unclassified|s__Bacilli_unclassified_SGB6571|t__SGB6571,...,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Alcaligenaceae|g__Kerstersia|s__Kerstersia_gyiorum|t__SGB13160,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_mediterraneense|t__SGB21135,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Eubacteriaceae|g__GGB28645|s__GGB28645_SGB41267|t__SGB41267,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27885|s__GGB27885_SGB40319|t__SGB40319,k__Archaea|p__Candidatus_Thermoplasmatota|c__Thermoplasmata|o__Methanomassiliicoccales|f__Methanomassiliicoccaceae|g__Methanomassiliicoccus|s__Methanomassiliicoccus_luminyensis|t__SGB33442,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae|g__GGB28250|s__GGB28250_SGB40793|t__SGB40793,k__Bacteria|p__Bacteroidetes|c__CFGB4425|o__OFGB4425|f__FGB4425|g__GGB27924|s__GGB27924_SGB40362|t__SGB40362,k__Bacteria|p__Actinobacteria|c__Actinobacteria|o__Propionibacteriales|f__Propionibacteriaceae|g__Propionibacteriaceae_unclassified|s__Propionibacteriaceae_bacterium_NML_150081|t__SGB15922,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27883|s__GGB27883_SGB40317|t__SGB40317,k__Bacteria|p__Bacteroidetes|c__CFGB570|o__OFGB570|f__FGB570|g__GGB1201|s__GGB1201_SGB1566|t__SGB1566
MMRS62666492ST-27-0-0,11.54777,6.73408,5.34157,3.17368,2.98767,2.71605,2.62759,2.55274,2.30150,2.12654,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS51737257ST-27-0-0,0.00000,0.67540,0.00041,0.00000,0.00000,4.25769,0.78997,0.00000,1.33619,0.82690,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS38861560ST-27-0-0,0.00561,3.45553,1.53598,0.00000,0.70183,13.45227,2.19867,0.00000,0.57779,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS95674036ST-27-0-0,0.00000,0.00000,0.00000,0.00000,0.00000,18.39640,1.92548,0.95926,0.02935,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS67096717ST-27-0-0,0.00000,0.00133,0.00000,0.00000,0.00000,6.14089,2.10991,1.17805,1.09216,0.03785,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SID72242320_SF07,0.00000,0.00000,1.05172,0.00000,0.88860,5.90205,2.47706,0.07185,0.12916,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SID30920067_SF07,24.37621,0.00000,0.94246,0.00000,0.00000,3.48781,3.01311,0.00000,0.78934,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SID11486372_SF08,0.00000,0.00000,0.00000,0.00000,0.00000,0.01068,3.31002,0.00000,0.31211,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SI

In [4]:
def arcsinsqrt(x):
    return np.arcsin(math.sqrt((x) / 100))  

In [106]:
sgb_abundances_transf = sgb_abundances.applymap(arcsinsqrt)
sgb_abundances_transf = sgb_abundances_transf.transpose()
sgb_abundances_transf

,MMRS62666492ST-27-0-0,MMRS51737257ST-27-0-0,MMRS38861560ST-27-0-0,MMRS95674036ST-27-0-0,MMRS67096717ST-27-0-0,MMRS84085284ST-27-0-0,MMRS51307502ST-27-0-0,MMRS17963147ST-27-0-0,MMRS39582183ST-27-0-0,MMRS35808664ST-27-0-0,...,SID15785561_SF07,SID19401504_SF05,SID20269046_SF08,SID50492523_SF07,SID10502288_SF08,SID72242320_SF07,SID30920067_SF07,SID11486372_SF08,SID16913125_SF05,SID16853657_SF05
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,0.346726,0.000000,0.007490,0.0,0.000000,0.317359,0.004837,0.002828,0.232361,0.566341,...,0.0,0.0,0.113852,0.000000,0.000000,0.000000,0.516365,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,0.262506,0.082276,0.186978,0.0,0.003647,0.000000,0.039493,0.000000,0.000000,0.062179,...,0.0,0.0,0.167096,0.000000,0.000000,0.000000,0.000000,0.0,0.214315,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,0.233227,0.002025,0.124254,0.0,0.000000,0.000000,0.000000,0.000000,0.087202,0.000000,...,0.0,0.0,0.078572,0.000000,0.000000,0.102734,0.097234,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__GGB3226|s__GGB3226_SGB4260|t__SGB4260,0.179104,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,0.173721,0.000000,0.083874,0.0,0.000000,0.000000,0.096163,0.000000,0.030529,0.000000,...,0.0,0.0,0.040494,0.047263,0.151663,0.094406,0.000000,0.0,0.144396,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae|g__GGB28250|s__GGB28250_SGB40793|t__SGB40793,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Bacteroidetes|c__CFGB4425|o__OFGB4425|f__FGB4425|g__GGB27924|s__GGB27924_SGB40362|t__SGB40362,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Actinobacteria|c__Actinobacteria|o__Propionibacteriales|f__Propionibacteriaceae|g__Propionibacteriaceae_unclassified|s__Propionibacteriaceae_bacterium_NML_150081|t__SGB15922,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0


In [6]:
ab_thresh = 0.001
keep = list()
for x in abundances.index.tolist():
    keep.append('t__' in x)
sgb_prevalences = abundances.loc[keep]
sgb_prevalences = sgb_prevalences.drop(sgb_prevalences.loc[sgb_prevalences.sum(axis=1) == 0].index.tolist())
sgb_prevalences[sgb_prevalences >= ab_thresh] = 1
sgb_prevalences[sgb_prevalences < ab_thresh] = 0
sgb_prevalences

,MMRS62666492ST-27-0-0,MMRS51737257ST-27-0-0,MMRS38861560ST-27-0-0,MMRS95674036ST-27-0-0,MMRS67096717ST-27-0-0,MMRS84085284ST-27-0-0,MMRS51307502ST-27-0-0,MMRS17963147ST-27-0-0,MMRS39582183ST-27-0-0,MMRS35808664ST-27-0-0,...,SID15785561_SF07,SID19401504_SF05,SID20269046_SF08,SID50492523_SF07,SID10502288_SF08,SID72242320_SF07,SID30920067_SF07,SID11486372_SF08,SID16913125_SF05,SID16853657_SF05
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__GGB3226|s__GGB3226_SGB4260|t__SGB4260,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae|g__GGB28250|s__GGB28250_SGB40793|t__SGB40793,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
k__Bacteria|p__Bacteroidetes|c__CFGB4425|o__OFGB4425|f__FGB4425|g__GGB27924|s__GGB27924_SGB40362|t__SGB40362,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
k__Bacteria|p__Actinobacteria|c__Actinobacteria|o__Propionibacteriales|f__Propionibacteriaceae|g__Propionibacteriaceae_unclassified|s__Propionibacteriaceae_bacterium_NML_150081|t__SGB15922,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
metadata = pd.read_csv('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_metadata.tsv', sep='\t')
metadata

,study_name,sample_id,subject_id,body_site,antibiotics_current_use,study_condition,disease,age,age_category,gender,...,non_westernized,sequencing_platform,PMID,number_reads,number_bases,minimum_read_length,median_read_length,NCBI_accession,curator,DNA_extraction_kit
0,NielsenHB_2014,O2_UC49_0,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,60584637.0,4.099972e+09,30.0,70.0,ERR210493;ERR210492;ERR209641;ERR209640,Paolo_Manghi,NaN
1,NielsenHB_2014,O2_UC49_2,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,56508214.0,3.775477e+09,30.0,70.0,ERR210495;ERR210494;ERR209643;ERR209642,Paolo_Manghi,NaN
2,NielsenHB_2014,O2_UC50_0,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,65105855.0,4.392995e+09,30.0,71.0,ERR210497;ERR210496;ERR209645;ERR209644,Paolo_Manghi,NaN
3,NielsenHB_2014,O2_UC50_2,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,53260679.0,3.596382e+09,30.0,70.0,ERR210500;ERR209648,Paolo_Manghi,NaN
4,NielsenHB_2014,O2_UC51_0,O2_UC51,stool,NaN,control,healthy,32.0,adult,female,...,no,IlluminaHiSeq,24997787.0,57486896.0,3.923766e+09,30.0,72.0,ERR210502;ERR210501;ERR209650;ERR209649,Paolo_Manghi,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12865,BorryM_2020,SAMEA6415059,SAMEA6415059,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12866,BorryM_2020,ERR3761407,ERR3761407,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12867,BorryM_2020,ERR3761411,ERR3761411,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12868,HaganRW_2019,Zape5,Zape5,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Fisher test

In [8]:
nw_samples = metadata[metadata['non_westernized'] == 'yes']['sample_id'].tolist()
w_samples = metadata[metadata['non_westernized'] == 'no']['sample_id'].tolist()

In [84]:
results = list()
pvalues = list()

for sgb in sgb_prevalences.index.tolist():
    nw = sgb_prevalences.loc[sgb][nw_samples].tolist()
    w = sgb_prevalences.loc[sgb][w_samples].tolist()
    if max([w.count(1) * 100 / len(w),  nw.count(1) * 100 / len(nw)]) > 20:
        odds, p = fisher_exact([[nw.count(0), w.count(0)], [w.count(1), nw.count(1)]])
        sgb_name = sgb.split('|')[-2] + '|' + sgb.split('|')[-1]
        res = {'odds_ration': odds, 'p-value': p, 'Westernized_prevalence': w.count(1) * 100 / len(w), 'Non-Westernized_prevalence': nw.count(1) * 100 / len(nw)}
        results.append(pd.Series(data=res, index=res.keys(), name=sgb_name))
        pvalues.append(p)
merged_project = pd.concat(results, axis=1).transpose()
merged_project['direction'] = merged_project['Westernized_prevalence'] > merged_project['Non-Westernized_prevalence']

In [85]:
_, qvalues = fdrcorrection(pvalues, method='poscorr')

In [86]:
merged_project['FDR'] = qvalues.tolist()

In [87]:
significant_sgbs = merged_project[merged_project['FDR']< 0.05]
significant_sgbs

,odds_ration,p-value,Westernized_prevalence,Non-Westernized_prevalence,direction,FDR
s__Prevotella_copri_clade_A|t__SGB1626,0.014648,0.000000e+00,37.642722,82.537835,False,0.000000e+00
s__Ruminococcus_sp_NSJ_71|t__SGB4290,0.040698,0.000000e+00,17.558213,44.470314,False,0.000000e+00
s__Coprococcus_eutactus|t__SGB5117,0.027080,0.000000e+00,32.581138,47.438882,False,0.000000e+00
s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,0.023652,0.000000e+00,40.753394,60.302678,False,0.000000e+00
s__Phocaeicola_vulgatus|t__SGB1814,0.036626,0.000000e+00,80.005394,56.635623,True,0.000000e+00
...,...,...,...,...,...,...
s__Lentisphaeria_bacterium|t__SGB9209,0.479845,5.374540e-09,0.872067,22.409779,False,5.555704e-09
s__Ruminococcaceae_bacterium|t__SGB14905,0.347141,2.173146e-23,1.375528,27.066356,False,2.315785e-23
s__GGB1250_SGB1673|t__SGB1673,0.474200,4.951833e-09,0.854086,21.420256,False,5.128352e-09
s__Clostridia_unclassified_SGB13999|t__SGB13999,0.286832,3.002790e-33,1.537355,23.923166,False,3.262874e-33


In [88]:
significant_sgbs.to_csv('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_fisher_test.tsv', sep='\t')

### Linear model for rel. abundance

In [9]:
sgb_abundances_transf

,MMRS62666492ST-27-0-0,MMRS51737257ST-27-0-0,MMRS38861560ST-27-0-0,MMRS95674036ST-27-0-0,MMRS67096717ST-27-0-0,MMRS84085284ST-27-0-0,MMRS51307502ST-27-0-0,MMRS17963147ST-27-0-0,MMRS39582183ST-27-0-0,MMRS35808664ST-27-0-0,...,SID15785561_SF07,SID19401504_SF05,SID20269046_SF08,SID50492523_SF07,SID10502288_SF08,SID72242320_SF07,SID30920067_SF07,SID11486372_SF08,SID16913125_SF05,SID16853657_SF05
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,0.346726,0.000000,0.007490,0.0,0.000000,0.317359,0.004837,0.002828,0.232361,0.566341,...,0.0,0.0,0.113852,0.000000,0.000000,0.000000,0.516365,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,0.262506,0.082276,0.186978,0.0,0.003647,0.000000,0.039493,0.000000,0.000000,0.062179,...,0.0,0.0,0.167096,0.000000,0.000000,0.000000,0.000000,0.0,0.214315,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,0.233227,0.002025,0.124254,0.0,0.000000,0.000000,0.000000,0.000000,0.087202,0.000000,...,0.0,0.0,0.078572,0.000000,0.000000,0.102734,0.097234,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__GGB3226|s__GGB3226_SGB4260|t__SGB4260,0.179104,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,0.173721,0.000000,0.083874,0.0,0.000000,0.000000,0.096163,0.000000,0.030529,0.000000,...,0.0,0.0,0.040494,0.047263,0.151663,0.094406,0.000000,0.0,0.144396,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae|g__GGB28250|s__GGB28250_SGB40793|t__SGB40793,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Bacteroidetes|c__CFGB4425|o__OFGB4425|f__FGB4425|g__GGB27924|s__GGB27924_SGB40362|t__SGB40362,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
k__Bacteria|p__Actinobacteria|c__Actinobacteria|o__Propionibacteriales|f__Propionibacteriaceae|g__Propionibacteriaceae_unclassified|s__Propionibacteriaceae_bacterium_NML_150081|t__SGB15922,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0


In [102]:
aux = sgb_abundances.loc[:,(sgb_abundances.astype(bool).sum(axis=0) > 12864*0.1).tolist()]
sgb_abundances_transf = aux.loc[~(aux==0).all(axis=1)].transpose().apply(multiplicative_replacement).apply(clr)
sgb_abundances_transf

,MMRS62666492ST-27-0-0,MMRS51737257ST-27-0-0,MMRS38861560ST-27-0-0,MMRS95674036ST-27-0-0,MMRS67096717ST-27-0-0,MMRS84085284ST-27-0-0,MMRS51307502ST-27-0-0,MMRS17963147ST-27-0-0,MMRS39582183ST-27-0-0,MMRS35808664ST-27-0-0,...,SID15785561_SF07,SID19401504_SF05,SID20269046_SF08,SID50492523_SF07,SID10502288_SF08,SID72242320_SF07,SID30920067_SF07,SID11486372_SF08,SID16913125_SF05,SID16853657_SF05
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,7.915685,-2.356604,0.624588,-1.009900,-2.415099,9.106642,-0.274998,-0.933025,7.350045,9.284715,...,-1.602480,-1.260498,6.073674,-1.369226,-2.396166,-2.243933,9.227255,-1.898595,-1.414936,-1.434671
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,7.376373,5.707426,7.047768,-1.009900,-0.855102,-1.343412,3.924001,-1.995384,-2.777064,4.973162,...,-1.602480,-1.260498,6.836021,-1.369226,-2.396166,-2.243933,-2.127517,-1.898595,8.272301,-1.434671
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,7.144712,-1.699477,6.236961,-1.009900,-2.415099,-1.343412,-2.614273,-1.995384,5.405403,-2.262692,...,-1.602480,-1.260498,5.334173,-1.369226,-2.396166,6.183371,5.974386,-1.898595,-1.414936,-1.434671
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,6.563686,-2.356604,5.453728,-1.009900,-2.415099,-1.343412,5.701266,-1.995384,3.308499,-2.262692,...,-1.602480,-1.260498,4.009976,5.302505,6.723378,6.014836,-2.127517,-1.898595,7.490906,-1.434671
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__Phocaeicola|s__Phocaeicola_vulgatus|t__SGB1814,6.468371,7.548603,8.406940,10.097824,7.582444,1.468275,3.575149,8.130174,6.386390,7.022889,...,5.270776,9.615234,7.352484,9.221449,6.120612,7.908244,7.282922,1.910290,7.362117,8.556159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Lachnospiraceae_unclassified|s__Lachnospiraceae_bacterium|t__SGB4834,-2.750397,-2.356604,-2.476188,-1.009900,-2.415099,-1.343412,-2.614273,-1.995384,-2.777064,-2.262692,...,-1.602480,-1.260498,-0.230558,-1.369226,-2.396166,-2.243933,-2.127517,-1.898595,-1.414936,-1.434671
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_D|t__SGB1636_group,-2.750397,-2.356604,-2.476188,-1.009900,-2.415099,-1.343412,-2.614273,-1.995384,-2.777064,-2.262692,...,-1.602480,-1.260498,-2.504063,-1.369226,-2.396166,-2.243933,-2.127517,-1.898595,-1.414936,-1.434671
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_SGB1653|t__SGB1653,-2.750397,-2.356604,-2.476188,-1.009900,-2.415099,-1.343412,-2.614273,-1.995384,-2.777064,-2.262692,...,-1.602480,-1.260498,-2.504063,-1.369226,-2.396166,-2.243933,-2.127517,-1.898595,-1.414936,-1.434671


In [116]:
metadata2 = metadata[metadata['non_westernized'] != 'ancient']
metadata2.index = metadata2['sample_id']
metadata2 = metadata2.dropna(axis=1, thresh=len(metadata2)*0.5)
metadata2 = metadata2.dropna(axis=0, how='any')

res = dict()
pvalues = list()
for index in sgb_abundances_transf.index:    
    nw = sgb_prevalences.loc[index][nw_samples].tolist()
    w = sgb_prevalences.loc[index][w_samples].tolist()
    if max([w.count(1) * 100 / len(w),  nw.count(1) * 100 / len(nw)]) > 20:    
        species_ab = pd.DataFrame(sgb_abundances_transf.loc[index])
        species_real_ab = pd.DataFrame(sgb_abundances.transpose().loc[index])
        species_ab = species_ab.rename(columns={index: 'rel_abundance'})
        species_real_ab = species_real_ab.rename(columns={index: 'rel_abundance'})
        for i in species_ab.index:
            if i not in metadata2.index.tolist():
                species_ab = species_ab.drop(i, axis=0)
                species_real_ab = species_real_ab.drop(i, axis=0)
        species_ab['non_westernized'] = metadata2.loc[species_ab.index]['non_westernized']
        species_real_ab['non_westernized'] = metadata2.loc[species_real_ab.index]['non_westernized']
        species_ab['gender'] = metadata2.loc[species_ab.index]['gender']
        species_ab['age_category'] = metadata2.loc[species_ab.index]['age_category']
        species_ab['number_reads'] = metadata2.loc[species_ab.index]['number_reads']
        species_ab['study_name'] = metadata2.loc[species_ab.index]['study_name']
        #species_ab = species_ab[species_ab['rel_abundance'] > 0]
        #species_real_ab = species_real_ab[species_real_ab['rel_abundance'] > 0]
        res_1 = smf.mixedlm(formula='rel_abundance ~ C(non_westernized) + number_reads + C(gender) + C(age_category)', data=species_ab, groups=species_ab['study_name']).fit() 
        p_lm = res_1.pvalues['C(non_westernized)[T.yes]']
        beta_lm = res_1.params['C(non_westernized)[T.yes]']
        omega_est = float(np.sqrt(res_1.cov_re.values[0,0]+(0.5*res_1.scale)))
        effect_size = res_1.params.loc["C(non_westernized)[T.yes]"]/omega_est
        ab_nw = species_real_ab[species_real_ab['non_westernized'] == 'yes']['rel_abundance'].values.tolist()
        ab_w = species_real_ab[species_real_ab['non_westernized'] == 'no']['rel_abundance'].values.tolist()
        pvalues.append(p_lm)
        res[index.split('|')[-2] + '|' + index.split('|')[-1]] = [effect_size, beta_lm, p_lm, w.count(1) * 100 / len(w), nw.count(1) * 100 / len(nw), statistics.mean(ab_w) < statistics.mean(ab_nw), statistics.mean(ab_w), statistics.mean(ab_nw), statistics.median(ab_w), statistics.median(ab_nw)]

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parame

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimiz

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.w

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 5.351829
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the b

...

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 1.475578
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
 

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.w

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 5.757564
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/base/model.py:604: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
 

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parame

/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/shares/CIBIO-Storage/CM/cmstore/tools/anaconda3/envs/aitor_jupyter/lib/python3.8/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parame

In [115]:
res

{'s__Phocaeicola_vulgatus|t__SGB1814': [-1.2424766900134725,
  -0.1486108091450692,
  0.003198170486129741,
  80.00539422817585,
  56.63562281722934,
  False,
  3.7261381610082305,
  0.07511636363636363,
  0.5724,
  0.0]}

In [117]:
res_1.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                 Mixed Linear Model Regression Results
=======================================================================
Model:                MixedLM     Dependent Variable:     rel_abundance
No. Observations:     4031        Method:                 REML         
No. Groups:           18          Scale:                  0.0004       
Min. group size:      1           Log-Likelihood:         9809.8754    
Max. group size:      1106        Converged:              Yes          
Mean group size:      223.9                                            
-----------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------
Intercept                     0.011    0.005  2.324 0.020  0.002  0.020
C(non_westernized)[T.yes]     0.039    0.013  2.903 0.004  0.013  0.066
C(gender)[T.male]             0.001    0.001  1.415 0.157 -0.000  0.002
C(age_category)[T.child]     -0.018    0.004 -4.566 0.000 -0.026 -0.010
C(age_category)[T.newborn]   -0.017    0.004 -4.320 0.000 -0.025 -0.009
C(age_category)[T.schoolage] -0.009    0.004 -2.256 0.024 -0.017 -0.001
C(age_category)[T.senior]    -0.003    0.003 -0.948 0.343 -0.009  0.003
number_reads                 -0.000    0.000 -1.421 0.155 -0.000  0.000
Group Var                     0.000    0.006                           
=======================================================================

"""

In [14]:
_, qvalues = fdrcorrection(pvalues, method='poscorr')

In [17]:
rel_ab_res = pd.DataFrame(res).transpose()
rel_ab_res.columns = ['beta' ,'p-value', 'prev_w', 'prev_nw', 'direction', 'mean_w', 'mean_nw', 'median_w', 'median_nw']
rel_ab_res['FDR'] = qvalues.tolist()

In [18]:
rel_ab_res

,beta,p-value,prev_w,prev_nw,direction,mean_w,mean_nw,median_w,median_nw,FDR
s__Prevotella_copri_clade_A|t__SGB1626,0.131655,0.148563,37.642722,82.537835,True,3.282836,6.48921,0.0,2.00552,0.285737
s__Ruminococcus_sp_NSJ_71|t__SGB4290,-0.006084,0.490893,17.558213,44.470314,False,0.374778,0.141259,0.0,0.0,0.633114
s__Coprococcus_eutactus|t__SGB5117,0.00549,0.608227,32.581138,47.438882,False,0.303356,0.179011,0.0,0.01719,0.722236
s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,-0.008156,0.215893,40.753394,60.302678,False,0.163507,0.073219,0.0,0.00269,0.362754
s__Phocaeicola_vulgatus|t__SGB1814,-0.148611,0.003198,80.005394,56.635623,False,3.726138,0.075116,0.5724,0.0,0.015198
...,...,...,...,...,...,...,...,...,...,...
s__Lentisphaeria_bacterium|t__SGB9209,0.011385,0.000263,0.872067,22.409779,True,0.001346,0.187986,0.0,0.0,0.001839
s__Ruminococcaceae_bacterium|t__SGB14905,0.005285,0.000002,1.375528,27.066356,True,0.000377,0.02959,0.0,0.0,0.000029
s__GGB1250_SGB1673|t__SGB1673,0.006071,0.003221,0.854086,21.420256,True,0.002934,0.081681,0.0,0.0,0.015198
s__Clostridia_unclassified_SGB13999|t__SGB13999,0.011461,0.0,1.537355,23.923166,True,0.004942,0.053722,0.0,0.0,0.000002


In [19]:
significant_sgbs = rel_ab_res[rel_ab_res['FDR'] < 0.05]
significant_sgbs

,beta,p-value,prev_w,prev_nw,direction,mean_w,mean_nw,median_w,median_nw,FDR
s__Phocaeicola_vulgatus|t__SGB1814,-0.148611,0.003198,80.005394,56.635623,False,3.726138,0.075116,0.5724,0.0,0.015198
s__Bacteroides_stercoris|t__SGB1830_group,-0.059389,0.012479,43.369595,18.277066,False,0.732226,0.008496,0.0,0.0,0.047507
s__Bacteroides_uniformis|t__SGB1836_group,-0.129046,0.000827,82.97222,52.677532,False,2.524605,0.136677,0.65269,0.0,0.004856
s__Clostridia_unclassified_SGB4373|t__SGB4373,0.006866,0.000001,4.692979,29.394645,True,0.010163,0.06067,0.0,0.0,0.000012
s__Parabacteroides_distasonis|t__SGB1934,-0.047718,0.002258,76.049627,51.629802,False,0.741537,0.049361,0.15542,0.00072,0.011330
...,...,...,...,...,...,...,...,...,...,...
s__Lentisphaeria_bacterium|t__SGB9209,0.011385,0.000263,0.872067,22.409779,True,0.001346,0.187986,0.0,0.0,0.001839
s__Ruminococcaceae_bacterium|t__SGB14905,0.005285,0.000002,1.375528,27.066356,True,0.000377,0.02959,0.0,0.0,0.000029
s__GGB1250_SGB1673|t__SGB1673,0.006071,0.003221,0.854086,21.420256,True,0.002934,0.081681,0.0,0.0,0.015198
s__Clostridia_unclassified_SGB13999|t__SGB13999,0.011461,0.0,1.537355,23.923166,True,0.004942,0.053722,0.0,0.0,0.000002


In [22]:
significant_sgbs.to_csv('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_rel_ab_dif_groups.tsv', sep='\t')

In [39]:
max(significant_sgbs['mean_nw'])

8.642841818181818